# Chapter 16 &mdash; SAT, 2-SAT and 3-SAT: Literals, Clauses, CNF

**Concept 5 of the Chapter 16 decomposition:** *SAT, 2-SAT and 3-SAT: Literals, Clauses, CNF*

CNF satisfiability, with exactly two or exactly three literals per clause.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-SAT-2SAT-3SAT/Concept-SAT-2SAT-3SAT.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The vocabulary, fixed once:

* a **literal** is a variable or its negation: $x_3$, $\neg x_3$;
* a **clause** is a disjunction of literals: $(x_1 \vee \neg x_2 \vee x_5)$;
* a formula in **conjunctive normal form (CNF)** is a conjunction of clauses;
* **$k$-SAT** restricts every clause to exactly $k$ literals.

Every Boolean formula has an equivalent CNF, and every CNF has an **equisatisfiable**
3-CNF (Concept 12) &mdash; so restricting to 3-SAT loses nothing.

The surprise is where the line falls: **2-SAT is in $P$** (Concept 6) while **3-SAT is
NP-complete** (Concept 8). One literal per clause is the difference between easy and
hardest-in-NP.

## 2. Definitions

### The CNF toolkit

In [ ]:
# --- a tiny CNF toolkit -------------------------------------------------
# A literal is an int: 3 means x3, -3 means NOT x3.
# A clause is a tuple of literals; a formula is a list of clauses.
from itertools import product

def nvars(F):
    return max((abs(l) for c in F for l in c), default=0)

def evaluate(F, assign):
    # assign: dict var -> bool
    return all(any(assign[abs(l)] == (l > 0) for l in c) for c in F)

def brute_sat(F):
    n = nvars(F)
    for bits in product([False, True], repeat=n):
        a = {i + 1: bits[i] for i in range(n)}
        if evaluate(F, a): return a
    return None

def show_cnf(F):
    def lit(l): return ("x%d" % l) if l > 0 else ("~x%d" % -l)
    return " AND ".join("(" + " OR ".join(lit(l) for l in c) + ")" for c in F)

### Random $k$-SAT instances, for experimenting

In [ ]:
import random
def random_ksat(k, nvar, nclause, seed=0):
    random.seed(seed)
    F = []
    while len(F) < nclause:
        vs = random.sample(range(1, nvar + 1), k)
        F.append(tuple(v * random.choice([1, -1]) for v in vs))
    return F

## 3. Tests

A formula, and a satisfying assignment.

In [ ]:
F = [(1, -2, 3), (-1, 2, 3), (1, 2, -3)]
print(show_cnf(F))
a = brute_sat(F)
print("satisfying assignment :", a)
assert a and evaluate(F, a)

An **unsatisfiable** formula: all eight clauses over three variables.

In [ ]:
U = [tuple(v * s for v, s in zip((1, 2, 3), signs))
     for signs in product([1, -1], repeat=3)]
print(show_cnf(U))
print("satisfiable? ", brute_sat(U) is not None)
assert brute_sat(U) is None
print("\nEvery assignment falsifies exactly one clause -- and there are eight.")

2-SAT, 3-SAT and general SAT, side by side.

In [ ]:
for k in [2, 3]:
    F = random_ksat(k, 5, 8, seed=3)
    a = brute_sat(F)
    print("random %d-SAT, 5 vars, 8 clauses : %s"
          % (k, "SAT " + str(sorted(a.items())) if a else "UNSAT"))

The satisfiability **threshold**: random 3-SAT flips near 4.26 clauses per variable.

In [ ]:
import random
n = 10
print("%-8s %-10s" % ("m/n", "fraction satisfiable"))
for ratio in [1, 2, 3, 4, 5, 6]:
    sat = 0
    for s in range(20):
        F = random_ksat(3, n, ratio * n, seed=s)
        if brute_sat(F): sat += 1
    print("%-8.1f %-10.2f" % (ratio, sat / 20.0))
print("\nThe drop is sharp, and near ratio 4.26 in the limit -- one of the")
print("most-studied phase transitions in computer science.")

Every clause size is representable; 3 is the interesting one.

In [ ]:
print("1-SAT : trivial -- each clause forces its literal")
print("2-SAT : polynomial (Concept 6)")
print("3-SAT : NP-complete (Concept 8)")
print("k-SAT for k>3 : NP-complete, and reducible to 3-SAT")
one = [(1,), (2,), (-3,)]
print("\n1-SAT example :", show_cnf(one), "->", brute_sat(one))
assert brute_sat(one) == {1: True, 2: True, 3: False}

## 4. Exercises


1. Convert $(x_1 \wedge x_2) \vee x_3$ to CNF by hand.
2. How many distinct clauses are there over $n$ variables with exactly 3 literals?
3. Why is 1-SAT trivial? Where does the triviality stop?

In [ ]:
# Your work for the exercises above.